In [17]:
import os
DOTENV_PATH = '../../apis/.env'
import dotenv
dotenv.load_dotenv(DOTENV_PATH)
hf_token_write = os.getenv('HF_TOKEN_WRITE')
def mask_token(token):
    return token[:4] + '*' * (len(token) - 8) + token[-4:]
print(f"HF_TOKEN_WRITE: {mask_token(hf_token_write)}")

from sentence_transformers import SentenceTransformer

from sentence_transformers.models import StaticEmbedding
from datasets import load_dataset
import duckdb
from typing import List

import time
def play_chimes():
    sound_path = r"C:\Windows\Media\chimes.wav"
    os.system(f'powershell -c (New-Object Media.SoundPlayer "{sound_path}").PlaySync();')
play_chimes() # Play the sound when this function runs (I use this to signal the end of long tasks)

HF_TOKEN_WRITE: hf_u*****************************Xipx


In [2]:
static_embedding = StaticEmbedding.from_model2vec("minishlab/potion-base-8M")
model = SentenceTransformer(modules=[static_embedding])

In [18]:
# ds = load_dataset("ai-blueprint/fineweb-bbc-news")
ds = load_dataset("reddgr/talking-to-chatbots-unwrapped-chats")

We can now create embeddings for the dataset. Normally, we might want to chunk our data into smaller batches to avoid losing precision, but for this example, we will just create embeddings for the full text of the dataset.

In [19]:
def create_embeddings(batch, column):
    # Ensure all entries are strings
    texts = [str(x) if x is not None else "" for x in batch[column]]
    embeddings = model.encode(texts, convert_to_numpy=True)
    batch[f"{column}-embeddings"] = embeddings.tolist()
    return batch

# ds = ds.map(lambda batch: create_embeddings(batch, column="text"), batched=True)
ds = ds.map(lambda batch: create_embeddings(batch, column="prompt"), batched=True)

play_chimes()

Map:   0%|          | 0/10774 [00:00<?, ? examples/s]

In [20]:
ds.push_to_hub("reddgr/talking-to-chatbots-prompts-embeddings", token = hf_token_write)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings/commit/627556490bf334d99d19988cc6d719037a7419f3', commit_message='Upload dataset', commit_description='', oid='627556490bf334d99d19988cc6d719037a7419f3', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='reddgr/talking-to-chatbots-prompts-embeddings'), pr_revision=None, pr_num=None)

In [30]:
ds = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")
print(ds)
ds["train"] = ds["train"].add_column("embeddings", ds["train"]["prompt-embeddings"])
print(ds)

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'prompt-embeddings'],
        num_rows: 10774
    })
})
DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'turn', 'prompt', 'response', 'category', 'language', 'pred_label_rq', 'prob_rq', 'pred_label_tl', 'prob_tl', 'model', 'message_tag', 'date', 'turns', 'source', 'chatbot_id', 'chatbot_name', 'attachments', 'conversation_tag', 'prompt-embeddings', 'embeddings'],
        num_rows: 10774
    })
})


In [31]:
ds.push_to_hub("reddgr/talking-to-chatbots-prompts-embeddings", token = hf_token_write)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/3.29k [00:00<?, ?B/s]

c:\Users\david\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\david\.cache\huggingface\hub\datasets--reddgr--talking-to-chatbots-prompts-embeddings. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


CommitInfo(commit_url='https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings/commit/26ce9629b3217402ff2bc492f95914c46051141e', commit_message='Upload dataset', commit_description='', oid='26ce9629b3217402ff2bc492f95914c46051141e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/reddgr/talking-to-chatbots-prompts-embeddings', endpoint='https://huggingface.co', repo_type='dataset', repo_id='reddgr/talking-to-chatbots-prompts-embeddings'), pr_revision=None, pr_num=None)

In [33]:
ttcb_dataset = load_dataset("reddgr/talking-to-chatbots-prompts-embeddings")["train"]
test_dataset_df = ttcb_dataset.to_pandas()
display(test_dataset_df[['prompt', 'embeddings']].sample(6))

,prompt,embeddings
7476,now we want to inner join two dataframes df1 a...,"[2.0681819915771484, 1.000893235206604, -2.311..."
1567,"print(head(sort(-my_vector), 5))\n\nthis helps...","[-1.7503056526184082, 1.0717812776565552, -1.1..."
2993,Recommend me a song,"[-5.088220596313477, -3.5961110591888428, -4.1..."
6426,is there a way to get the size of an object wi...,"[-1.6899466514587402, 0.5473678112030029, -2.5..."
1898,Who is your author?,"[-1.0496736764907837, 0.44597119092941284, -1...."
8445,Context code:\n\n# Parse the HTML\r\nsoup = Be...,"[-0.45370402932167053, 0.4114408493041992, -3...."


In [ ]:
def similarity_search_without_duckdb_index(
    query: str,
    k: int = 5,
    dataset_name: str = "reddgr/talking-to-chatbots-prompts-embeddings",
    embedding_column: str = "embeddings",
):
    # Use same model as used for indexing
    query_vector = model.encode(query)
    embedding_dim = model.get_sentence_embedding_dimension()

    sql = f"""
        SELECT 
            *,
            array_cosine_distance(
                {embedding_column}::float[{embedding_dim}], 
                {query_vector.tolist()}::float[{embedding_dim}]
            ) as distance
        FROM 'hf://datasets/{dataset_name}/**/*.parquet'
        ORDER BY distance
        LIMIT {k}
    """
    return duckdb.sql(sql).to_df()

similarity_search_without_duckdb_index("What is the future of AI?")

InvalidInputException: Invalid Input Error: No magic bytes found at end of file 'hf://datasets/reddgr/talking-to-chatbots-prompts-embeddings/data/train-00000-of-00001.parquet'

In [19]:
def _setup_vss():
    duckdb.sql(
        query="""
        INSTALL vss;
        LOAD vss;
        """
    )


def _drop_table(table_name):
    duckdb.sql(
        query=f"""
        DROP TABLE IF EXISTS {table_name};
        """
    )


def _create_table(dataset_name, table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE TABLE {table_name} AS 
        SELECT *, {embedding_column}::float[{model.get_sentence_embedding_dimension()}] as {embedding_column}_float 
        FROM 'hf://datasets/{dataset_name}/**/*.parquet';
        """
    )


def _create_index(table_name, embedding_column):
    duckdb.sql(
        query=f"""
        CREATE INDEX my_hnsw_index ON {table_name} USING HNSW ({embedding_column}_float) WITH (metric = 'cosine');
        """
    )


def create_index(dataset_name, table_name, embedding_column):
    _setup_vss()
    _drop_table(table_name)
    _create_table(dataset_name, table_name, embedding_column)
    _create_index(table_name, embedding_column)


create_index(
    dataset_name="ai-blueprint/fineweb-bbc-news-embeddings",
    table_name="fineweb_bbc_news_embeddings",
    embedding_column="embeddings",
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
def similarity_search_with_duckdb_index(
    query: str, k: int = 5, table_name: str = "fineweb_bbc_news_embeddings", embedding_column: str = "embeddings"
):
    embedding = model.encode(query).tolist()
    return duckdb.sql(
        query=f"""
        SELECT *, array_cosine_distance({embedding_column}_float, {embedding}::FLOAT[{model.get_sentence_embedding_dimension()}]) as distance 
        FROM {table_name}
        ORDER BY distance 
        LIMIT {k};
    """
    ).to_df()


similarity_search_with_duckdb_index("What is love?")

,url,text,embeddings,embeddings_float,distance
0,http://news.bbc.co.uk/cbbcnews/hi/newsid_17700...,February 14 is Valentine's Day.\nIt's named af...,"[-1.6512084007263184, -0.2739017605781555, -1....","[-1.6512084, -0.27390176, -1.5980083, 2.005432...",0.286753
1,http://news.bbc.co.uk/cbbcnews/hi/newsid_17700...,February 14 is Valentine's Day.\nIt's named af...,"[-1.6512084007263184, -0.2739017605781555, -1....","[-1.6512084, -0.27390176, -1.5980083, 2.005432...",0.286753
2,https://www.bbc.co.uk/news/magazine-24223786,A Point of View: Putting a price on love\nOur ...,"[-3.3412487506866455, 0.11048950254917145, -1....","[-3.3412488, 0.1104895, -1.4662294, 0.5659724,...",0.450999
3,https://www.bbc.com/news/in-pictures-43060122,"Your pictures: Valentine's Day\nEach week, we ...","[0.03464483842253685, -0.9412625432014465, -1....","[0.03464484, -0.94126254, -1.2021598, 0.427660...",0.468012
4,http://www.bbc.co.uk/news/magazine-24379830,Why is a children's book about rabbits being r...,"[-1.4426814317703247, -0.7511618733406067, -1....","[-1.4426814, -0.7511619, -1.336863, 1.4392253,...",0.476307
